# Notebook 8 - Recommendation Engine

**The capstone notebook.** Everything we've built since Notebook 01 now combines into a deliverable: a per-customer recommendation that marketing can act on.

**Input (from all prior notebooks):**
- `data/processed/customer_segments.parquet` - RFM segments + K-Means clusters (Notebook 04)
- `data/processed/clv_predictions.parquet` - predicted 90-day revenue per-customer (Notebook 05)
- `data/processed/churn_predictions.parquet` - churn risk + strategy quadrant (Notebook 06)
- `data/processed/product_cross_sell.parquet` - products cross-sell rules (Notebook 07)
- `data/interim/transactions_customer_level.parquet` - customer purchase history
- `models/clv_xgboost.joblib`, `models/churn_xgboost.joblib` - trained models with SHAP

**Output:**
- `data/processed/recommendations.parquet` - one row par customer with their complete recommendation
- `data/processed/recommendations.jsonl` - same data as JSON lines, ready for an API or marketing automation system
- `data/customer_recommendation_examples.md` - formatted markdown samples for manager/boss pitch.

## The architecture
For each customer, we produce a rich JSON record:
```
{
    customer_id: 12345,
    segment: 'At Risk',
    strategy_quadrant: 'Urgent Win-back',
    predicted_clv_90d: 312.45,
    churn_risk: 0.68,
    top_churn_drivers: ['recency_day (281d)', 'revenue_last_90d (£0)'],
    recommended_action: 'Personalized win-back email with 15% discount',
    recommended_products: [
        {code: '85099B', desc: 'JUMBO BAG STRAWBERRY', lift: 6.43},
        ...
    ],
    expected_left_value: 47.20,
    priority_score: 0.78
}
```

**Why this matters**: the strategy matrix told us WHO to target. SHAP tells us WHY. Market basket tells us WHAT to offer. This notebook combines all three into a unit that marketing automation can consume directly.

## Notebook structure:
1. Load all inputs
2. Cuild the customer master table (one row per customer with everything joined)
3. Generate SHAP-driven explanations
4. Map each customer to their best product recommendations.
5. Define the action playbook (segment → tactics)
6. Compute priority scores for marketing ranking
7. Build final per-customer recommendation records
8. Save as parquet + JSONL
9. Produce formatted examples for the managers pitch

## Setup

In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import json
import joblib
from pathlib import Path
import shap

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', '{:,.2f}'.format)

INTERIM_PATH = Path('../data/interim')
PROCESSED_PATH = Path('../data/processed')
MODELS_PATH = Path('../models')
REPORTS_PATH = Path('../reports/figures')
REPORTS_PATH.mkdir(parents=True, exist_ok=True)

## 1. Load all inputs
Every prior notebook's output is now a building block. Verify everything loads cleanly before building the joins.

In [2]:
segment = pd.read_parquet(PROCESSED_PATH / 'customer_segments.parquet')
clv = pd.read_parquet(PROCESSED_PATH / 'clv_predictions.parquet')
churn = pd.read_parquet(PROCESSED_PATH / 'churn_predictions.parquet')
cross_sell = pd.read_parquet(PROCESSED_PATH / 'product_cross_sell.parquet')
transactions = pd.read_parquet(INTERIM_PATH / 'transactions_customer_level.parquet')

churn_artifact = joblib.load(MODELS_PATH / 'churn_xgboost.joblib')
churn_model = churn_artifact['model']
churn_features = churn_artifact['feature_names']

print(f'Segment:    {len(segment):,} customers')
print(f'CLV preds:  {len(clv):,} customers')
print(f'Churn preds:{len(churn):,} customers')
print(f'Cross-sell: {len(cross_sell):,} product-recommendation pairs ({cross_sell["product_code"].nunique():,} unique source products)')
print(f'Transactions: {len(transactions):,} rows')
print(f'Churn model has {len(churn_features)} features')


Segment:    5,256 customers
CLV preds:  5,256 customers
Churn preds:5,256 customers
Cross-sell: 57 product-recommendation pairs (30 unique source products)
Transactions: 802,637 rows
Churn model has 46 features


## 2. Build the customer master table
Join segments + CLV + churn into one row per customer. This is the foundation everything else attaches to.

In [3]:
master = (
    segment[['CustomerID', 'segment', 'recency_days', 'frequency', 'monetary', 'is_one_time_customer', 'kmeans_cluster', 'cluster_name']]
    .merge(
        clv[['CustomerID', 'predicted_clv_90d', 'clv_decile']],
        on='CustomerID', how='left'
    )
    .merge(
        churn[['CustomerID', 'churn_risk', 'p_active', 'strategy_quadrant']],
        on='CustomerID', how='left'
    )
)

print(f'Master table: {master.shape[0]:,} customers x {master.shape[1]} columns')
master.head(3)

Master table: 5,256 customers x 13 columns


,CustomerID,segment,recency_days,frequency,monetary,is_one_time_customer,kmeans_cluster,cluster_name,predicted_clv_90d,clv_decile,churn_risk,p_active,strategy_quadrant
0,12346.0,Loyal Customers,235,3,"77,352.96",0,3,Top-Tier,9.10,5,0.68,0.32,2. Urgent Win-back
1,12347.0,Champions,39,6,"4,114.18",0,3,Top-Tier,395.21,1,0.12,0.88,1. VIP Retention
2,12348.0,Loyal Customers,158,4,"1,388.40",0,0,Mid-Tier Active,36.74,3,0.31,0.69,1. VIP Retention
